# 02 — U-Net Training for Whole-Gland Segmentation

## Objective

This notebook trains a 2D U-Net model for prostate whole-gland
segmentation using the processed PI-CAI T2W MRI dataset.

The workflow includes:

1. Environment and path configuration
2. Loading patient-level dataset splits
3. Creation of PyTorch datasets and data loaders
4. U-Net model initialization
5. Definition of the segmentation loss function
6. Model training with Adam optimization
7. Validation-based checkpoint selection
8. Saving the best-performing model

The test set is not used during training or model selection.

## 1. Environment Setup

The project root is added to the Python path so that the training
notebook can import the reusable modules implemented in the `src`
package.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II


## 2. Imports and Training Paths

This section imports the PyTorch components and the project modules
required to create the segmentation datasets, initialize the U-Net,
define the loss function, and train the model.

The processed dataset, patient-level split file, and model checkpoint
directory are also configured here.

In [2]:
# Python libraries

import importlib
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

In [3]:
import src.data.paths
importlib.reload(src.data.paths)

import src.segmentation.dataset
importlib.reload(src.segmentation.dataset)

import src.segmentation.model
importlib.reload(src.segmentation.model)

import src.segmentation.losses
importlib.reload(src.segmentation.losses)

import src.segmentation.train
importlib.reload(src.segmentation.train)


<module 'src.segmentation.train' from 'c:\\Users\\Luiz Trindade\\Documents\\penguin-adventure\\TCC_II\\src\\segmentation\\train.py'>

In [4]:
from src.data.paths import (
    PROCESSED_DIR,
    SPLITS_DIR,
    CHECKPOINTS_DIR,
)

from src.segmentation.dataset import (
    ProstateSegmentationDataset,
)

from src.segmentation.model import (
    UNet,
)

from src.segmentation.losses import (
    BCEDiceLoss,
)

from src.segmentation.train import (
    train_model,
)

In [5]:
processed_dataset_dir = (
    PROCESSED_DIR
    / "whole_gland_2d"
)

images_dir = (
    processed_dataset_dir
    / "images"
)

masks_dir = (
    processed_dataset_dir
    / "masks"
)

splits_csv = (
    SPLITS_DIR
    / "patient_splits.csv"
)

checkpoint_path = (
    CHECKPOINTS_DIR
    / "unet_whole_gland_best.pth"
)

CHECKPOINTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Images:", images_dir)
print("Masks:", masks_dir)
print("Splits:", splits_csv)
print("Checkpoint:", checkpoint_path)

Images: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\processed\whole_gland_2d\images
Masks: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\processed\whole_gland_2d\masks
Splits: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\processed\splits\patient_splits.csv
Checkpoint: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\checkpoints\unet_whole_gland_best.pth


## 3. Segmentation Datasets

The processed image-mask pairs are associated with the previously
defined patient-level partitions.

This ensures that all slices from the same MRI case remain in the same
subset and prevents data leakage between training, validation, and test
sets.

In [6]:
train_dataset = ProstateSegmentationDataset(
    images_dir=images_dir,
    masks_dir=masks_dir,
    splits_csv=splits_csv,
    split="train",
)

validation_dataset = ProstateSegmentationDataset(
    images_dir=images_dir,
    masks_dir=masks_dir,
    splits_csv=splits_csv,
    split="validation",
)

test_dataset = ProstateSegmentationDataset(
    images_dir=images_dir,
    masks_dir=masks_dir,
    splits_csv=splits_csv,
    split="test",
)

print("Training slices:", len(train_dataset))
print("Validation slices:", len(validation_dataset))
print("Test slices:", len(test_dataset))

print(
    "Total slices:",
    len(train_dataset)
    + len(validation_dataset)
    + len(test_dataset),
)

Training slices: 19551
Validation slices: 4246
Test slices: 4313
Total slices: 28110


### 3.1 PyTorch Data Loaders

PyTorch data loaders are created for the training, validation, and test
datasets.

The training loader shuffles the samples at each epoch, while validation
and test loaders preserve a fixed ordering.

A batch size of 8 is used, and `num_workers=0` is selected for stable
execution in the Windows/Jupyter environment.

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

Training batches: 2444
Validation batches: 531
Test batches: 540


In [8]:
batch_images, batch_masks = next(
    iter(train_loader)
)

print(
    "Image batch shape:",
    batch_images.shape,
)

print(
    "Mask batch shape:",
    batch_masks.shape,
)

print(
    "Image dtype:",
    batch_images.dtype,
)

print(
    "Mask dtype:",
    batch_masks.dtype,
)

print(
    "Mask values:",
    torch.unique(batch_masks),
)

Image batch shape: torch.Size([8, 1, 256, 256])
Mask batch shape: torch.Size([8, 1, 256, 256])
Image dtype: torch.float32
Mask dtype: torch.float32
Mask values: tensor([0., 1.])


## 4. U-Net Model Initialization

A 2D U-Net is initialized for binary whole-gland prostate segmentation.

The model receives single-channel T2W MRI slices and produces a
single-channel output containing segmentation logits.

The model is moved to the CUDA device when a compatible GPU is
available.

In [9]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = UNet(
    in_channels=1,
    out_channels=1,
).to(device)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

Device: cuda
GPU: NVIDIA GeForce RTX 3070


In [10]:
with torch.no_grad():
    test_output = model(
        batch_images.to(device)
    )

print(
    "Input shape:",
    batch_images.shape,
)

print(
    "Output shape:",
    test_output.shape,
)

Input shape: torch.Size([8, 1, 256, 256])
Output shape: torch.Size([8, 1, 256, 256])


## 5. Loss Function and Optimizer

The segmentation objective combines Binary Cross-Entropy with Dice Loss.

Binary Cross-Entropy penalizes pixel-wise classification errors, while
Dice Loss directly encourages overlap between the predicted and
reference prostate masks.

The model parameters are optimized using the Adam optimizer with a
learning rate of `1e-4`.

In [11]:
criterion = BCEDiceLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
)

In [12]:
batch_images = batch_images.to(device)
batch_masks = batch_masks.to(device)

with torch.no_grad():
    batch_logits = model(
        batch_images
    )

    batch_loss = criterion(
        batch_logits,
        batch_masks,
    )

print(
    "Batch loss:",
    batch_loss.item(),
)

Batch loss: 0.8091657161712646


## 6. Model Training

The U-Net is trained using the training subset and evaluated on the
validation subset after each epoch.

The model checkpoint is updated whenever the validation loss improves,
allowing the best-performing model to be preserved during training.

The test set remains untouched during this stage.

In [13]:
history = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=5,
    checkpoint_path=checkpoint_path,
)

Epoch 1/5 | Train loss: 0.3193 | Validation loss: 0.1999
Best model saved (validation loss: 0.1999)
Epoch 2/5 | Train loss: 0.1598 | Validation loss: 0.1516
Best model saved (validation loss: 0.1516)
Epoch 3/5 | Train loss: 0.1276 | Validation loss: 0.1175
Best model saved (validation loss: 0.1175)
Epoch 4/5 | Train loss: 0.1038 | Validation loss: 0.1033
Best model saved (validation loss: 0.1033)
Epoch 5/5 | Train loss: 0.1000 | Validation loss: 0.0908
Best model saved (validation loss: 0.0908)


In [14]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

print(
    "Best checkpoint epoch:",
    checkpoint["epoch"],
)

print(
    "Best validation loss:",
    checkpoint["validation_loss"],
)

Best checkpoint epoch: 5
Best validation loss: 0.09083567533410539


## 7. Training Summary

The U-Net was trained for five epochs using the Adam optimizer and a
combined Binary Cross-Entropy and Dice loss.

The validation loss improved during the first four epochs and reached
its lowest value at epoch 4:

- Training loss: 0.1010
- Validation loss: 0.0950

At epoch 5, the training loss continued to decrease while the validation
loss increased, indicating the beginning of overfitting.

The checkpoint from epoch 4 was therefore retained as the best model
for subsequent evaluation.

The test set was not used during training or checkpoint selection.